# Mature Trendlines Research Lab

Full package-local research workbench. It prepares explicit data, runs tested causal replay, exposes deterministic diagnostics, and opens read-only TVLC viewers.

## 0. Methodology and Safety

Default run is SMOKE + SYNTHETIC. No provider call, YAML mutation, adequacy score, promotion decision, or notebook-owned model loop occurs. All model, replay, evidence, and viewer work stays in tested package APIs.

In [ ]:
from datetime import datetime, timezone

import pandas as pd
from IPython.display import IFrame, display

from libs.models.trendlines.research_lab import (
    TrendlineReplayWindow,
    lab_config_table,
    lab_controls_table,
    lab_export_table,
    lab_identity_table,
    lab_line_table,
    lab_performance_table,
    lab_pivot_count_table,
    lab_position_comparison_table,
    lab_ray_table,
    lab_replay_summary_table,
    lab_selected_pivot_table,
    lab_signal_history_table,
    lab_signal_table,
    lab_snapshot_timeline,
    lab_source_table,
    lab_study_registry_table,
    run_research_lab,
    select_replay_position,
    synthetic_lab_controls,
)

## 1. Research Controls

Controls contain research scope, explicit source bounds, replay windows, viewer policy, and provider authorization only. Model parameters remain YAML-resolved.

In [ ]:
controls = synthetic_lab_controls(
    asset="BTCUSDT",
    timeframes=("1h", "4h"),
    primary_timeframe="1h",
    seed=7,
    start_time=datetime(2025, 1, 1, tzinfo=timezone.utc),
    bar_counts={"1h": 48, "4h": 48},
    replay_windows={
        "1h": TrendlineReplayWindow(19, 20, 47, 1),
        "4h": TrendlineReplayWindow(19, 20, 47, 1),
    },
    include_signals=True,
    viewer_lookback_bars=32,
    start_inline_viewers=True,
    permanent_export=False,
)
display(lab_controls_table_placeholder := pd.DataFrame([controls.to_dict()]))

## 2. Data Specification

Synthetic data is deterministic and bounded. Injected and Binance examples remain explicit and disabled in this smoke run.

Disabled injected example:
```python
controls = injected_lab_controls(...)
injected_frames = {"1h": frame_1h, "4h": frame_4h}
frame = read_research_frame_artifact(...)
```

Disabled Binance example:
```python
controls = binance_lab_controls(..., provider_calls_authorized=True)
```

In [ ]:
data_specification = controls.data_spec.to_dict()
injected_example = {"mode": "injected", "frames": {"1h": "caller-supplied frame"}}
binance_example = {"mode": "binance", "requires": ["research purpose", "explicit loader", "provider authorization"]}
data_specification, injected_example, binance_example

## 3. Source and Availability Manifest

Preparation records event bounds, exact availability bounds, source identity, availability identity, timestamp semantics, and provenance for every timeframe.

In [ ]:
lab_session = await run_research_lab(controls)
session_result = lab_session
source_manifest = lab_session.time_table(lab_source_table, lab_session)
display(source_manifest)

## 4. Resolved YAML Configuration

Resolved extractor, fitter, and parameter mappings are displayed from the prepared configuration bundle. No constructor-default lookup occurs in the notebook.

In [ ]:
resolved_config_table = lab_session.time_table(lab_config_table, lab_session)
display(resolved_config_table)

## 5. Preparation Identities

Full identities bind prepared data and configuration. Timing values are intentionally excluded from identities.

In [ ]:
identity_table = lab_session.time_table(lab_identity_table, lab_session)
preparation_id = lab_session.preparation_id
dataset_id = lab_session.dataset_id
research_configuration_id = lab_session.research_configuration_id
replay_id = lab_session.replay_id
display(identity_table)

## 6. Multi-Timeframe Causal Replay

Each timeframe executes confirmed prefixes independently. Warm-up and intermediate positions update state even when recording stride changes.

In [ ]:
replay_windows = controls.replay_spec.to_dict()
executed_positions = {
    timeframe: lab_session.replay.timeframes[timeframe].executed_positions
    for timeframe in controls.timeframes
}
replay_windows

## 7. Replay Summary Dashboard

Summary rows remain descriptive: valid/invalid observations, geometry counts, signals, and temporal bounds. No adequacy judgement is made here.

In [ ]:
replay_summary_table = lab_session.time_table(lab_replay_summary_table, lab_session)
display(replay_summary_table)

## 8. Per-Timeframe TVLC Viewers

One package-local loopback viewer is opened per requested timeframe. The viewer consumes validated payloads and never executes model work.

In [ ]:
for timeframe in controls.timeframes:
    selected = lab_session.selections[timeframe]
    viewer_manifest = {
        "timeframe": timeframe,
        "selected_position": selected.position,
        "selection_reason": selected.selection_reason,
        "event_at": selected.point.event_at.isoformat(),
        "available_at": selected.point.available_at.isoformat(),
        "finality": selected.snapshot_row.finality,
        "viewer_url": lab_session.viewer_urls[timeframe],
    }
    display(lab_session.time_table(pd.DataFrame, [viewer_manifest]))
    display(IFrame(src=lab_session.viewer_urls[timeframe], width="100%", height=640))

## 9. Replay Position Navigator

Edit these variables to select another recorded position. Navigation rebuilds selected evidence and payload without rerunning replay.

In [ ]:
NAV_TIMEFRAME = "1h"
navigator_recorded_positions = lab_session.replay.timeframes[NAV_TIMEFRAME].recorded_positions
NAV_POSITION = navigator_recorded_positions[-2] if len(navigator_recorded_positions) > 1 else navigator_recorded_positions[-1]
NAV_LOOKBACK = 32
replay_id_before_navigation = lab_session.replay_id
navigator_viewer_url = lab_session.open_viewer(NAV_TIMEFRAME, NAV_POSITION, lookback=NAV_LOOKBACK)
selection = lab_session.selections[NAV_TIMEFRAME]
assert lab_session.replay_id == replay_id_before_navigation
assert selection.viewer_payload["selected_position"] == NAV_POSITION
selection_summary = {
    "timeframe": selection.timeframe,
    "position": selection.position,
    "reason": selection.selection_reason,
    "event_at": selection.point.event_at.isoformat(),
    "available_at": selection.point.available_at.isoformat(),
    "viewer_payload_id": selection.viewer_payload["payload_id"],
    "replay_id_unchanged": lab_session.replay_id == replay_id_before_navigation,
    "viewer_url": navigator_viewer_url,
}
display(selection_summary)
display(lab_session.time_table(lab_selected_pivot_table, selection))
display(lab_session.time_table(lab_line_table, selection))
display(lab_session.time_table(lab_ray_table, selection))
display(lab_session.time_table(lab_signal_table, selection))
display(IFrame(src=navigator_viewer_url, width="100%", height=640))
selection_summary

## 10. Position-to-Position Comparison

Comparison reports exact identity, state, geometry, pivot, quality, and signal differences. It does not call changes improvements or deterioration.

In [ ]:
recorded = lab_session.replay.timeframes[NAV_TIMEFRAME].recorded_positions
LEFT_POSITION = recorded[-2] if len(recorded) > 1 else recorded[-1]
RIGHT_POSITION = recorded[-1]
position_comparison_table = lab_session.time_table(lab_position_comparison_table,
    lab_session,
    timeframe=NAV_TIMEFRAME,
    left_position=LEFT_POSITION,
    right_position=RIGHT_POSITION,
)
display(position_comparison_table)

## 11. Pivot Diagnostics

Pivot counts use authoritative pipeline metadata. Selected pivot rows come from the approved selected-position diagnostic API.

In [ ]:
pivot_count_table = lab_session.time_table(lab_pivot_count_table, lab_session, NAV_TIMEFRAME)
selected_pivot_table = lab_session.time_table(lab_selected_pivot_table, selection)
display(pivot_count_table.tail())
display(selected_pivot_table)

## 12. Fitted-Line Diagnostics

Line rows retain role, ordinal, geometry, point identity, and evidence identity from the canonical replay.

In [ ]:
line_table = lab_session.time_table(lab_line_table, selection)
display(line_table)

## 13. Boundary-Ray Quality

Ray rows expose canonical geometry and quality values without extrapolation or notebook-side fitting.

In [ ]:
ray_table = lab_session.time_table(lab_ray_table, selection)
display(ray_table)

## 14. Native Signals and Interactions

Signals and interactions are read from selected native output. Signal history remains paired by snapshot, revision, and knowledge time.

In [ ]:
signal_table = lab_session.time_table(lab_signal_table, selection)
signal_history_table = lab_session.time_table(lab_signal_history_table, selection)
display(signal_table)
display(signal_history_table)

## 15. Snapshot, Revision and Knowledge Timeline

The timeline keeps event time and availability/knowledge time distinct and retains full replay-point identities.

In [ ]:
timeline_table = lab_session.time_table(lab_snapshot_timeline, lab_session, NAV_TIMEFRAME)
display(timeline_table)

## 16. Performance Diagnostics

Timing values are operational diagnostics only. They never participate in preparation, replay, evidence, or viewer identities.

In [ ]:
performance_table = lab_session.time_table(lab_performance_table, lab_session)
display(performance_table)

## 17. Evidence and Viewer Export

Permanent export is disabled in this checked-in smoke run. Explicit export controls can write validated evidence bundles, viewer bundles, and one lab manifest.

In [ ]:
export_table = lab_session.time_table(lab_export_table, lab_session)
display(export_table)

## 18. Research Study Registry

Available descriptive studies are separated from L2-D adequacy work and the separate oscillator programme. No placeholder scores are fabricated.

In [ ]:
study_registry_table = lab_session.time_table(lab_study_registry_table, lab_session)
display(study_registry_table)

## 19. Final Status and Cleanup

Close every viewer and remove temporary bundles. This cell is required for clean top-to-bottom execution.

In [ ]:
provider_calls_made = lab_session.provider_calls_made
viewer_timeframes_before_cleanup = tuple(lab_session.viewer_sessions)
owned_temporary_roots = tuple(path.parent for path in lab_session.viewer_bundle_paths.values())
lab_session.close()
viewer_servers_closed = not lab_session.viewer_sessions
temporary_bundles_removed = all(not root.exists() for root in owned_temporary_roots)
final_status = {
    "status": "SMOKE_COMPLETE",
    "provider_calls_made": provider_calls_made,
    "viewer_servers_closed": viewer_servers_closed,
    "temporary_bundles_removed": temporary_bundles_removed,
    "owned_temporary_roots": [str(root) for root in owned_temporary_roots],
    "permanent_exports": {key: str(value) for key, value in lab_session.export_paths.items()},
}
final_status